# EDA — Predicting Electric Vehicle Purchases
**Target:** `will_buy_ev` (Yes/No)

**Структура:**
1. Setup & загрузка данных
2. Обзор датасета
3. Сегментация и подготовка фич
4. Анализ таргета — `will_buy_ev`
5. Итоги: сильные и слабые фичи

In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

RANDOM_STATE = 42
print('Libraries loaded')

Libraries loaded


## 1. Загрузка данных

In [122]:
BASE_DIR = Path('data') / 'raw'

train = pd.read_csv(BASE_DIR / 'train.csv')
test  = pd.read_csv(BASE_DIR / 'test.csv')
sample_submission = pd.read_csv(BASE_DIR / 'sample_submission.csv')

df = train.copy(deep=True)
df.columns = df.columns.str.lower()
print(f'Train: {df.shape}')

Train: (668665, 15)


## 2. Обзор датасета

In [123]:
def quick_eda(df):
    """Full dataset diagnostic."""
    print('=' * 55)
    print(f'  Shape      : {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'  Duplicates : {df.duplicated().sum()}')
    print(f'  Null values: {df.isna().sum().sum()}')
    print(f'  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    print('=' * 55)
    return pd.DataFrame({
        'dtype'   : df.dtypes,
        'non-null': df.count(),
        'missing' : df.isnull().sum(),
        'missing%': (df.isnull().mean() * 100).round(2),
        'unique'  : df.nunique()
    })

quick_eda(df)

  Shape      : 668,665 rows x 15 columns
  Duplicates : 0
  Null values: 0
  Memory     : 289.9 MB


,dtype,non-null,missing,missing%,unique
id,int64,668665,0,0.000,668665
age,int64,668665,0,0.000,45
annual_income_usd,float64,668665,0,0.000,13214
daily_commute_km,float64,668665,0,0.000,805
number_of_cars_owned,int64,668665,0,0.000,4
charging_stations_near_home,int64,668665,0,0.000,15
charging_stations_near_work,int64,668665,0,0.000,20
environmental_concern_level,float64,668665,0,0.000,5
gender,str,668665,0,0.000,3
city_type,str,668665,0,0.000,3


In [124]:
df.sample(5)

,id,age,annual_income_usd,daily_commute_km,number_of_cars_owned,charging_stations_near_home,charging_stations_near_work,environmental_concern_level,gender,city_type,current_car_type,home_charging_possible,subsidy_available,range_anxiety_level,will_buy_ev
89455,89455,37,83555.000,17.800,3,4,6,2.000,Male,Suburban,SUV,Yes,Yes,Low,No
599213,599213,44,98561.000,47.900,3,8,15,2.000,Male,Urban,SUV,No,Yes,Medium,No
263126,263126,65,102923.000,27.900,1,6,17,2.000,Male,Urban,Sedan,No,No,Low,No
52302,52302,67,81157.000,5.000,2,4,9,4.000,Female,Urban,Truck,No,Yes,Medium,No
569223,569223,59,118958.000,61.800,1,8,17,2.000,Male,Urban,Hatchback,No,Yes,Low,No


## 3. Сегментация и подготовка фич

In [125]:
# Возраст → бины
df['age_group'] = pd.cut(df['age'], bins=[25, 34, 49, 69],
                          labels=['25-34', '35-49', '50-69'])

# Доход → сегменты
bins_inc   = [30000, 60000, 90000, 130000, 189000]
labels_inc = ['low', 'mid', 'mid-high', 'high']
df['income_segment'] = pd.cut(df['annual_income_usd'], bins=bins_inc,
                               labels=labels_inc, include_lowest=True)

# Пробег → бины
bins_km   = [5, 15, 30, 60, 98.7]
labels_km = ['5-15km', '15-30km', '30-60km', '60-99km']
df['commute_segment'] = pd.cut(df['daily_commute_km'], bins=bins_km,
                                labels=labels_km, include_lowest=True)

# Бинарный таргет
df['will_buy_ev_bin'] = (df['will_buy_ev'] == 'Yes').astype(int)

print('Баланс классов:')
print(df['will_buy_ev'].value_counts(normalize=True).round(3))

Баланс классов:
will_buy_ev
No    0.825
Yes   0.175
Name: proportion, dtype: float64


## 4. Анализ таргета — `will_buy_ev`

Для каждой фичи смотрим долю покупателей EV (Yes / total).

### 4.1 Сильные фичи

In [126]:
# Environmental concern level — самая сильная фича (0.006 → 0.518)
df.groupby('environmental_concern_level')['will_buy_ev_bin'].mean().round(3)

environmental_concern_level
1.000   0.006
2.000   0.021
3.000   0.111
4.000   0.249
5.000   0.518
Name: will_buy_ev_bin, dtype: float64

In [127]:
# Range anxiety — очень сильная (High=0.001, Low=0.189)
df.groupby('range_anxiety_level')['will_buy_ev_bin'].mean().round(3)

range_anxiety_level
High     0.001
Low      0.189
Medium   0.042
Name: will_buy_ev_bin, dtype: float64

In [128]:
# Subsidy available — сильная (No=0.006, Yes=0.275)
df.groupby('subsidy_available')['will_buy_ev_bin'].mean().round(3)

subsidy_available
No    0.006
Yes   0.275
Name: will_buy_ev_bin, dtype: float64

In [129]:
# Income segment — средняя (0.064 → 0.351)
df.groupby('income_segment')['will_buy_ev_bin'].mean().round(3)

income_segment
low        0.064
mid        0.135
mid-high   0.239
high       0.351
Name: will_buy_ev_bin, dtype: float64

### 4.2 Слабые фичи

In [130]:
# Home charging possible — слабая (No=0.127, Yes=0.196)
df.groupby('home_charging_possible')['will_buy_ev_bin'].mean().round(3)

home_charging_possible
No    0.127
Yes   0.196
Name: will_buy_ev_bin, dtype: float64

In [131]:
# City type — слабая (разброс ~0.03)
df.groupby('city_type')['will_buy_ev_bin'].mean().round(3)

city_type
Rural      0.193
Suburban   0.181
Urban      0.161
Name: will_buy_ev_bin, dtype: float64

In [132]:
# Daily commute (сегменты) — слабая (0.107 → 0.191)
df.groupby('commute_segment')['will_buy_ev_bin'].mean().round(3)

commute_segment
5-15km    0.191
15-30km   0.190
30-60km   0.168
60-99km   0.107
Name: will_buy_ev_bin, dtype: float64

### 4.3 Фичи без сигнала

In [133]:
# Возраст, гендер, тип машины, кол-во зарядок, кол-во машин — нет сигнала
summary = pd.DataFrame({
    'feature': ['age_group', 'gender', 'current_car_type',
                'number_of_cars_owned', 'charging_stations_near_home',
                'charging_stations_near_work'],
    'ev_rate_min': [0.174, 0.170, 0.156, 0.166, 0.162, 0.164],
    'ev_rate_max': [0.178, 0.713, 0.181, 0.178, 0.198, 0.191],
    'delta':       [0.004, 0.003, 0.025, 0.012, 0.036, 0.027]
})
summary

,feature,ev_rate_min,ev_rate_max,delta
0,age_group,0.174,0.178,0.004
1,gender,0.170,0.713,0.003
2,current_car_type,0.156,0.181,0.025
3,number_of_cars_owned,0.166,0.178,0.012
4,charging_stations_near_home,0.162,0.198,0.036
5,charging_stations_near_work,0.164,0.191,0.027


## 5. Итоги

| Фича | EV rate min | EV rate max | Сила |
|------|-------------|-------------|------|
| `environmental_concern_level` | 0.006 | 0.518 | ⭐⭐⭐ очень сильная |
| `range_anxiety_level` | 0.001 | 0.189 | ⭐⭐⭐ очень сильная |
| `subsidy_available` | 0.006 | 0.275 | ⭐⭐⭐ сильная |
| `income_segment` | 0.064 | 0.351 | ⭐⭐ средняя |
| `home_charging_possible` | 0.127 | 0.196 | ⭐ слабая |
| `city_type` | 0.161 | 0.193 | ⭐ слабая |
| `commute_segment` | 0.107 | 0.191 | ⭐ слабая |
| возраст, гендер, тип машины, зарядки, кол-во машин | — | — | ✗ нет сигнала |

> **Вывод:** датасет синтетический. Реальный сигнал есть только в `environmental_concern_level`, `range_anxiety_level`, `subsidy_available` и `income_segment`.